### Consumer Pricing

Transactional records w/ premise and items

- https://data.gov.my/data-catalogue/pricecatcher
- https://data.gov.my/data-catalogue/lookup_item
- https://data.gov.my/data-catalogue/lookup_premise


In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install sklearn
%pip install scipy
%pip install pyarrow
%pip install fastparquet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import fastparquet
import pyarrow
import pyarrow.parquet as pq
import pyarrow.csv as pcsv
import pyarrow.ipc as pipc
import pyarrow.json as pjson
import pyarrow.orc as porc

In [2]:
# Source URLs from the catalog metadata pages
TRANSACTION_URL = "https://storage.data.gov.my/pricecatcher/pricecatcher_2026-02.parquet"
ITEM_URL = "https://storage.data.gov.my/pricecatcher/lookup_item.parquet"
PREMISE_URL = "https://storage.data.gov.my/pricecatcher/lookup_premise.parquet"


def load_dataset(parquet_url: str) -> pd.DataFrame:
    """Load parquet when available; fallback to CSV if parquet engine is missing."""
    try:
        return pd.read_parquet(parquet_url)
    except Exception:
        csv_url = parquet_url.replace(".parquet", ".csv")
        return pd.read_csv(csv_url)


# Load datasets
transaction_df = load_dataset(TRANSACTION_URL)
item_df = load_dataset(ITEM_URL)
premise_df = load_dataset(PREMISE_URL)

if "date" in transaction_df.columns:
    transaction_df["date"] = pd.to_datetime(
        transaction_df["date"], errors="coerce")

print("transaction_df:", transaction_df.shape)
print("item_df:", item_df.shape)
print("premise_df:", premise_df.shape)

transaction_df.head()

transaction_df: (1216577, 4)
item_df: (757, 5)
premise_df: (3838, 6)


,date,premise_code,item_code,price
0,2026-02-01,3,2,10.6
1,2026-02-01,3,88,14.9
2,2026-02-01,3,92,14.2
3,2026-02-01,3,94,18.0
4,2026-02-01,3,95,7.6


In [3]:
# Metadata (from data.gov.my catalog pages)
metadata = {
    "pricecatcher": {
        "description": "Transactional price records.",
        "variables": ["date", "premise_code", "item_code", "price"],
        "url": TRANSACTION_URL,
        "join_key": ["premise_code", "item_code"],
    },
    "lookup_item": {
        "description": "Item lookup table.",
        "variables": ["item_code", "item_name", "unit", "item_group", "item_category"],
        "url": ITEM_URL,
        "join_key": ["item_code"],
    },
    "lookup_premise": {
        "description": "Premise lookup table.",
        "variables": ["premise_code", "premise", "address", "premise_type", "state", "district"],
        "url": PREMISE_URL,
        "join_key": ["premise_code"],
    },
}

In [4]:
# Join to one raw table (left joins preserve all transactions)
joined_df = (
    transaction_df
    .merge(item_df, on="item_code", how="left", validate="many_to_one")
    .merge(premise_df, on="premise_code", how="left", validate="many_to_one")
)

print("joined_df:", joined_df.shape)
joined_df.head()

joined_df: (1216577, 13)


,date,premise_code,item_code,price,item,unit,item_group,item_category,premise,address,premise_type,state,district
0,2026-02-01,3,2,10.6,AYAM BERSIH - SUPER,1kg,BARANGAN SEGAR,AYAM,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
1,2026-02-01,3,88,14.9,IKAN KELI (ANTARA 2 HINGGA 5 EKOR SEKILOGRAM),1kg,BARANGAN SEGAR,IKAN DARAT,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
2,2026-02-01,3,92,14.2,CILI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
3,2026-02-01,3,94,18.0,CILI MERAH - MINYAK,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah
4,2026-02-01,3,95,7.6,HALIA BASAH (TUA),1kg,BARANGAN SEGAR,SAYUR-SAYURAN,JUSCO AYER KEROH,"LOT 4991,MUKIM BUKIT BARU,75450 LEBUH AYER KER...",Pasar Raya / Supermarket,Melaka,Melaka Tengah


In [5]:
# Save raw joined dataset (before cleaning/normalization)
os.makedirs("data", exist_ok=True)
raw_output_parquet = "data/pricecatcher_joined_raw.parquet"
raw_output_csv = "data/pricecatcher_joined_raw.csv"

try:
    joined_df.to_parquet(raw_output_parquet, index=False)
    print(f"Saved raw joined dataset to: {raw_output_parquet}")
except Exception:
    joined_df.to_csv(raw_output_csv, index=False)
    print(f"Parquet engine unavailable; saved CSV instead: {raw_output_csv}")

# Quick EDA snapshot
eda_summary = {
    "rows": len(joined_df),
    "columns": joined_df.shape[1],
    "duplicates": int(joined_df.duplicated().sum()),
}

print("EDA summary:", eda_summary)
print("\nDtypes:")
print(joined_df.dtypes.to_frame("dtype").to_string())
print("\nMissing values (%):")
print((joined_df.isna().mean() *
       100).sort_values(ascending=False).to_frame("missing_pct").to_string())

Saved raw joined dataset to: data/pricecatcher_joined_raw.parquet
EDA summary: {'rows': 1216577, 'columns': 13, 'duplicates': 0}

Dtypes:
                        dtype
date           datetime64[ns]
premise_code            int64
item_code               int64
price                 float64
item                   object
unit                   object
item_group             object
item_category          object
premise                object
address                object
premise_type           object
state                  object
district               object

Missing values (%):
               missing_pct
item             11.582004
unit             11.582004
item_group       11.582004
item_category    11.582004
date              0.000000
premise_code      0.000000
item_code         0.000000
price             0.000000
premise           0.000000
address           0.000000
premise_type      0.000000
state             0.000000
district          0.000000


### Filter for core analysis facet: `item_category`

For category-based analysis, rows without `item_category` are excluded. The missing share is about **11%**, so removing these records still preserves roughly **89%** of observations, which remains large enough to maintain strong coverage for downstream EDA.


In [6]:
# Drop rows with missing item_category (main analysis facet)
analysis_df = joined_df.dropna(subset=["item_category"]).copy()

rows_before = len(joined_df)
rows_after = len(analysis_df)
rows_dropped = rows_before - rows_after
pct_dropped = (rows_dropped / rows_before) * 100
pct_kept = 100 - pct_dropped

print(f"Rows before: {rows_before:,}")
print(f"Rows after : {rows_after:,}")
print(f"Dropped    : {rows_dropped:,} ({pct_dropped:.2f}%)")
print(f"Kept       : {pct_kept:.2f}%")

# EDA on transaction date range
earliest_date = analysis_df["date"].min()
latest_date = analysis_df["date"].max()
print("\nTransaction date range (after item_category filter):")
print(f"Earliest date: {earliest_date}")
print(f"Latest date  : {latest_date}")

# Missing values check after filtering
print("\nMissing values (%) after item_category filter:")
print((analysis_df.isna().mean() *
       100).sort_values(ascending=False).to_frame("missing_pct").to_string())

Rows before: 1,216,577
Rows after : 1,075,673
Dropped    : 140,904 (11.58%)
Kept       : 88.42%

Transaction date range (after item_category filter):
Earliest date: 2026-02-01 00:00:00
Latest date  : 2026-02-24 00:00:00

Missing values (%) after item_category filter:
               missing_pct
date                   0.0
premise_code           0.0
item_code              0.0
price                  0.0
item                   0.0
unit                   0.0
item_group             0.0
item_category          0.0
premise                0.0
address                0.0
premise_type           0.0
state                  0.0
district               0.0


### Cardinality review and column-reduction decisions

We assess cardinality for `item`, `unit`, and `price` to check whether these fields are highly variant or close to invariant for analysis purposes.

Planned reductions for the working analysis dataset:

- Drop `item_code` and `premise_code` because they are technical join keys.
- Drop `address` because it is too granular for the current level of analysis.

This keeps semantic business fields while removing keys and overly granular location text.


In [ ]:
# Cardinality analysis for selected columns
cardinality_cols = ["item", "unit", "price"]
cardinality_summary = pd.DataFrame(
    {
        "n_unique": analysis_df[cardinality_cols].nunique(dropna=True),
        "unique_ratio_pct": (analysis_df[cardinality_cols].nunique(dropna=True) / len(analysis_df)) * 100,
    }
).sort_values("n_unique", ascending=False)

print("Cardinality summary (after item_category filter):")
print(cardinality_summary.to_string())

print("Top 10 most frequent items:")
print(analysis_df["item"].value_counts().head(
    10).to_frame("count").to_string())

print("Top 10 most frequent units:")
print(analysis_df["unit"].value_counts().head(
    10).to_frame("count").to_string())

# Build reduced analysis dataset by dropping keys and overly granular location text
drop_cols = ["item_code", "premise_code", "address"]
analysis_reduced_df = analysis_df.drop(columns=drop_cols).copy()

print("\nReduced analysis dataset shape:", analysis_reduced_df.shape)
print("Dropped columns:", drop_cols)
print("Remaining columns:", list(analysis_reduced_df.columns))

Cardinality summary (after item_category filter):
       n_unique  unique_ratio_pct
price      2152          0.200061
item        268          0.024915
unit         59          0.005485
Top 10 most frequent items:
                             count
item                              
HALIA BASAH (TUA)            22987
BAWANG BESAR KUNING/HOLLAND  22850
BAWANG PUTIH IMPORT (CHINA)  22513
LOBAK MERAH                  21972
TIMUN                        21835
TOMATO                       21373
KUBIS BUNGA (CAULIFLOWER)    20525
UBI KENTANG IMPORT (CHINA)   20444
KUNYIT HIDUP                 19958
LENGKUAS                     19454
Top 10 most frequent units:
          count
unit           
1kg      767755
250 g     38852
30 biji   36099
5 kg      17554
2 kg      17090
500 g     16135
850g      15900
340 g     14502
425 g     10273
400 g      8720

Reduced analysis dataset shape: (1075673, 10)
Dropped columns: ['item_code', 'premise_code', 'address']
Remaining columns: ['date', 'price', 'ite

### Unit normalization and derived price metrics

Cleaning and feature-engineering steps:

1. Normalize `unit` by trimming leading/trailing whitespace and removing internal spaces between quantity and unit (for example, `340 g` -> `340g`, `5 kg` -> `5kg`).
2. Parse normalized `unit` into numeric quantity and metric token.
3. Convert only metric units in grams and kilograms into a new numeric column `unit_in_kg`:
   - `kg` -> unchanged quantity
   - `g` -> quantity / 1000
4. Leave `unit_in_kg` as null for non-conforming units (for example, `biji`/pieces or other unsupported tokens).
5. Create `price_per_kg` as `price / unit_in_kg` where `unit_in_kg` is available.
6. Report how many rows have null `unit_in_kg` because their unit is not conforming to `kg` or `g`.


In [8]:
# Normalize unit strings and derive unit_in_kg + price_per_kg
analysis_cleaned_df = analysis_reduced_df.copy()

# Step 1: normalize whitespace and casing in unit
analysis_cleaned_df["unit"] = (
    analysis_cleaned_df["unit"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
    .str.lower()
)

# Step 2: parse quantity and metric token from normalized unit
unit_parts = analysis_cleaned_df["unit"].str.extract(
    r"^(?P<unit_value>\d+(?:\.\d+)?)(?P<unit_metric>[a-zA-Z]+)$"
)

unit_value = pd.to_numeric(unit_parts["unit_value"], errors="coerce")
metric = unit_parts["unit_metric"].str.lower()

# Step 3: convert supported metrics to kg
metric_factor = {"kg": 1.0, "g": 0.001}
analysis_cleaned_df["unit_in_kg"] = unit_value * metric.map(metric_factor)

# Step 4: derive price_per_kg where unit_in_kg is valid
analysis_cleaned_df["price_per_kg"] = np.where(
    analysis_cleaned_df["unit_in_kg"].notna() & (
        analysis_cleaned_df["unit_in_kg"] > 0),
    analysis_cleaned_df["price"] / analysis_cleaned_df["unit_in_kg"],
    np.nan,
)

# Step 5: report non-conforming rows
null_unit_in_kg_count = int(analysis_cleaned_df["unit_in_kg"].isna().sum())
null_unit_in_kg_pct = (null_unit_in_kg_count / len(analysis_cleaned_df)) * 100

print("Unit normalization examples:")
print(
    analysis_cleaned_df[["unit", "unit_in_kg", "price", "price_per_kg"]]
    .drop_duplicates(subset=["unit"])
    .head(12)
    .to_string(index=False)
)

print(
    f"Rows with null unit_in_kg (non-conforming units): {null_unit_in_kg_count:,} ({null_unit_in_kg_pct:.2f}%)")

print("Top 15 non-conforming units:")
print(
    analysis_cleaned_df.loc[analysis_cleaned_df["unit_in_kg"].isna(), "unit"]
    .value_counts()
    .head(15)
    .to_frame("count")
    .to_string()
)

print("Missing values (%) after unit engineering:")
print((analysis_cleaned_df.isna().mean() *
       100).sort_values(ascending=False).to_frame("missing_pct").to_string())

Unit normalization examples:
  unit  unit_in_kg  price  price_per_kg
   1kg       1.000  10.60     10.600000
30biji         NaN  12.95           NaN
  425g       0.425  10.50     24.705882
  325g       0.325   4.20     12.923077
   5kg       5.000  30.50      6.100000
  250g       0.250   6.20     24.800000
  500g       0.500   4.30      8.600000
  340g       0.340   4.50     13.235294
   2kg       2.000  13.30      6.650000
 5x79g         NaN   5.40           NaN
  10kg      10.000  37.99      3.799000
+-450g         NaN   1.60           NaN
Rows with null unit_in_kg (non-conforming units): 114,181 (10.61%)
Top 15 non-conforming units:
          count
unit           
30biji    36099
1liter     8352
1.5liter   6707
200ml      6472
5x79g      5302
2liter     4808
320ml      4799
345ml      4740
150ml      4060
+-450g     3828
1batang    3295
340ml      3176
500ml      2175
25x18g     2107
100beg     2000
Missing values (%) after unit engineering:
               missing_pct
unit_in_kg   

### Second-pass unit parsing for complex patterns

To recover more rows for `unit_in_kg`, we apply additional parsing rules on units that were still non-conforming after the first pass:

1. Multipack pattern such as `5x79g` -> `5 * 79g`.
2. Signed/approximate prefix pattern such as `+-450g` -> `450g`.
3. Recalculate `unit_in_kg` for these recovered cases.
4. Recompute `price_per_kg`.
5. Report recovered row count and remaining non-conforming units.


In [9]:
# Second-pass parser to recover additional kg/g-convertible units
analysis_enhanced_df = analysis_cleaned_df.copy()

null_before_second_pass = int(analysis_enhanced_df["unit_in_kg"].isna().sum())

units = analysis_enhanced_df["unit"].astype(str).str.lower()

# Rule A: multipack, e.g. 5x79g, 25x18g
multi_parts = units.str.extract(
    r"^(?P<count>\d+)x(?P<size>\d+(?:\.\d+)?)(?P<metric>kg|g)$")
multi_count = pd.to_numeric(multi_parts["count"], errors="coerce")
multi_size = pd.to_numeric(multi_parts["size"], errors="coerce")
multi_metric = multi_parts["metric"]
multi_factor = multi_metric.map({"kg": 1.0, "g": 0.001})
multi_unit_in_kg = multi_count * multi_size * multi_factor

# Rule B: signed/approximate prefix, e.g. +-450g, +450g, -450g
signed_parts = units.str.extract(
    r"^[+\-]+(?P<value>\d+(?:\.\d+)?)(?P<metric>kg|g)$")
signed_value = pd.to_numeric(signed_parts["value"], errors="coerce")
signed_metric = signed_parts["metric"]
signed_factor = signed_metric.map({"kg": 1.0, "g": 0.001})
signed_unit_in_kg = signed_value * signed_factor

# Fill only previously null rows
analysis_enhanced_df["unit_in_kg"] = analysis_enhanced_df["unit_in_kg"].fillna(
    multi_unit_in_kg)
analysis_enhanced_df["unit_in_kg"] = analysis_enhanced_df["unit_in_kg"].fillna(
    signed_unit_in_kg)

# Recompute price_per_kg after second-pass recovery
analysis_enhanced_df["price_per_kg"] = np.where(
    analysis_enhanced_df["unit_in_kg"].notna() & (
        analysis_enhanced_df["unit_in_kg"] > 0),
    analysis_enhanced_df["price"] / analysis_enhanced_df["unit_in_kg"],
    np.nan,
)

null_after_second_pass = int(analysis_enhanced_df["unit_in_kg"].isna().sum())
recovered_rows = null_before_second_pass - null_after_second_pass

print(f"Null unit_in_kg before second pass: {null_before_second_pass:,}")
print(f"Null unit_in_kg after second pass : {null_after_second_pass:,}")
print(f"Recovered rows                    : {recovered_rows:,}")
print(
    f"Remaining null share              : {(null_after_second_pass / len(analysis_enhanced_df)) * 100:.2f}%")

print("\nTop 15 remaining non-conforming units:")
print(
    analysis_enhanced_df.loc[analysis_enhanced_df["unit_in_kg"].isna(), "unit"]
    .value_counts()
    .head(15)
    .to_frame("count")
    .to_string()
)

print("\nSample recovered rows:")
print(
    analysis_enhanced_df.loc[
        analysis_cleaned_df["unit_in_kg"].isna(
        ) & analysis_enhanced_df["unit_in_kg"].notna(),
        ["unit", "unit_in_kg", "price", "price_per_kg"]
    ].head(15)
    .to_string(index=False)
)

Null unit_in_kg before second pass: 114,181
Null unit_in_kg after second pass : 98,470
Recovered rows                    : 15,711
Remaining null share              : 9.15%

Top 15 remaining non-conforming units:
          count
unit           
30biji    36099
1liter     8352
1.5liter   6707
200ml      6472
2liter     4808
320ml      4799
345ml      4740
150ml      4060
1batang    3295
340ml      3176
500ml      2175
100beg     2000
25ml       1883
330ml      1695
1biji      1685

Sample recovered rows:
  unit  unit_in_kg  price  price_per_kg
 5x79g       0.395    5.4     13.670886
+-450g       0.450    1.6      3.555556
+-450g       0.450    1.8      4.000000
 5x79g       0.395    5.5     13.924051
+-450g       0.450    2.3      5.111111
 5x79g       0.395    6.2     15.696203
+-450g       0.450    2.0      4.444444
+-450g       0.450    1.6      3.555556
 5x79g       0.395    6.0     15.189873
+-450g       0.450    1.7      3.777778
 5x79g       0.395    5.0     12.658228
+-450g      